In [38]:
# Packages
import os
import re

# For downloading NOAA data
import gzip
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor
from selenium import webdriver
import time
import requests

# For data analysis
import pandas as pd
import numpy as np


In [ ]:
# Web scrape NOAA weather data links
storms_url = 'https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/'
req = requests.get(storms_url)                   # access url webpage
soup = BeautifulSoup(req.text, 'html.parser')    # parse thru HTML text of webpage

# Find 2000s data.csv.gz filenames in <a href="link" > format 
pattern = r'StormEvents_details-ftp_v1\.0_d20\d{2}_c\d{8}\.csv\.gz'    # regex filename pattern
data_links = []
for link in soup.find_all('a', attrs={'href': re.compile(pattern)}):   # find, combine year data w/ url
   year_data = link.get('href')
   full_link = str(storms_url) + str(year_data)
   data_links.append(full_link)

# Download gzip data to data/ directory
def download_files(data_urls):
    # Create new dirs for data
    data_dir = '../data'
    os.makedirs(data_dir, exist_ok=True)
    
    for data in data_urls:
        response = requests.get(data, stream=True)
        
        # Check in request for 'content-disposition' header to parse filenames
        if 'content-disposition' in response.headers:
            content_disp = response.headers['content-disposition']
            file_name = content_disp.split('filename=')[1]
        else:
            file_name = data.split('/')[-1]
        
        # Write downloaded gzip data to gzip dir
        gz_name = os.path.join(data_dir, file_name)
        with open(gz_name, 'wb') as gz_file:
            gz_file.write(response.content)
        
        print(f'Downloaded file to {gz_name} \n')

download_files(data_links[:2])

Downloaded file to ../data/StormEvents_details-ftp_v1.0_d2000_c20260323.csv.gz 

Downloaded file to ../data/StormEvents_details-ftp_v1.0_d2001_c20260323.csv.gz 



In [44]:
# Create new dirs for data
gzip_dir, csv_dir = '../data/gzip', '../data/csv'
os.makedirs(gzip_dir, exist_ok=True)
os.makedirs(csv_dir, exist_ok=True)